<a href="https://colab.research.google.com/github/Leopqs/Treinamento_de_IAs/blob/main/2bimestre/exercicios/Aula007.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os
import zipfile
import cv2
import numpy as np

from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score


# ============================
# CONFIGURAÇÕES
# ============================

ZIP_PATH = "imagens_formas-20260518T173408Z-3-001.zip"
PASTA_SAIDA = "imagens_formas_extraidas"
IMG_SIZE = 128


# ============================
# EXTRAIR ZIP
# ============================

if not os.path.exists(PASTA_SAIDA):
    os.makedirs(PASTA_SAIDA)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(PASTA_SAIDA)


# ============================
# PRÉ-PROCESSAMENTO
# ============================

def preprocessar_imagem(caminho):
    img = cv2.imread(caminho, cv2.IMREAD_GRAYSCALE)

    if img is None:
        raise ValueError(f"Erro ao carregar imagem: {caminho}")

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    # Suaviza ruídos
    img = cv2.GaussianBlur(img, (3, 3), 0)

    # Binarização automática
    img = cv2.adaptiveThreshold(
        img,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        21,
        8
    )

    return img


def extrair_caracteristicas(img):
    features = hog(
        img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys"
    )

    return features


# ============================
# AUMENTO DE DADOS
# ============================

def aumentar_imagem(img):
    imagens = [img]

    altura, largura = img.shape
    centro = (largura // 2, altura // 2)

    angulos = [-20, -10, 10, 20]

    for angulo in angulos:
        matriz = cv2.getRotationMatrix2D(centro, angulo, 1.0)
        rotacionada = cv2.warpAffine(img, matriz, (largura, altura))
        imagens.append(rotacionada)

    # Pequenos deslocamentos
    deslocamentos = [(-5, 0), (5, 0), (0, -5), (0, 5)]

    for dx, dy in deslocamentos:
        matriz = np.float32([[1, 0, dx], [0, 1, dy]])
        deslocada = cv2.warpAffine(img, matriz, (largura, altura))
        imagens.append(deslocada)

    return imagens


# ============================
# CARREGAR IMAGENS
# ============================

X = []
y = []

for raiz, pastas, arquivos in os.walk(PASTA_SAIDA):
    for arquivo in arquivos:
        if arquivo.lower().endswith((".jpg", ".jpeg", ".png")):
            caminho = os.path.join(raiz, arquivo)

            # Pega o nome da classe pelo nome do arquivo
            # Exemplo: quadrado_1.jpg -> quadrado
            classe = arquivo.split("_")[0].lower()

            img = preprocessar_imagem(caminho)

            imagens_aumentadas = aumentar_imagem(img)

            for img_aug in imagens_aumentadas:
                features = extrair_caracteristicas(img_aug)
                X.append(features)
                y.append(classe)


X = np.array(X)
y = np.array(y)


print("Total de amostras após aumento de dados:", len(X))
print("Classes encontradas:", sorted(set(y)))


# ============================
# TREINO E TESTE
# ============================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)


modelo = SVC(
    kernel="rbf",
    C=10,
    gamma="scale"
)

modelo.fit(X_train, y_train)


# ============================
# AVALIAÇÃO
# ============================

y_pred = modelo.predict(X_test)

print("Acurácia:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))


# ============================
# TESTAR UMA IMAGEM ESPECÍFICA
# ============================

def prever_forma(caminho_imagem):
    img = preprocessar_imagem(caminho_imagem)
    features = extrair_caracteristicas(img).reshape(1, -1)

    predicao = modelo.predict(features)

    return predicao[0]


resultado = prever_forma("imagens_formas_extraidas/imagens_formas/quadrado_1.jpg")
print("Forma prevista:", resultado)

Total de amostras após aumento de dados: 288
Classes encontradas: [np.str_('asterisco'), np.str_('circulo'), np.str_('estrela'), np.str_('hexagono'), np.str_('pentagono'), np.str_('quadrado'), np.str_('retangulo'), np.str_('triangulo')]
Acurácia: 0.8472222222222222

              precision    recall  f1-score   support

   asterisco       1.00      0.89      0.94         9
     circulo       0.89      0.89      0.89         9
     estrela       0.90      1.00      0.95         9
    hexagono       0.80      0.89      0.84         9
   pentagono       1.00      0.67      0.80         9
    quadrado       0.80      0.89      0.84         9
   retangulo       0.73      0.89      0.80         9
   triangulo       0.75      0.67      0.71         9

    accuracy                           0.85        72
   macro avg       0.86      0.85      0.85        72
weighted avg       0.86      0.85      0.85        72

Forma prevista: quadrado
